# `cuda_oracle.ipynb` — Kaggle CUDA authoritative oracle (BENCH-01 / BENCH-02)

**This is the authoritative GPU oracle of record.** It is a *human-gated external run*:
a human executes this notebook top-to-bottom on a **Kaggle CUDA** instance and records
the measured pass/fail + speed into `bench/RESULTS.md`. ROCm in-env is smoke-only — **not**
a gate.

## Harness order (correctness is a BLOCKING gate before any speed number)

1. Build the `--features cuda` wheel + `nvidia-smi` (confirm CUDA active).
2. **Standalone primitive oracles** (scan, segmented scan, radix sort / single-bit
   reorder, reduce-by-key, segmented-reduce, `update_part_props`) vs serial CPU refs —
   `<=1e-4`; integer/index primitives **bit-exact**.
3. **Bit-packed cindex oracle** vs CPU quantized layout (GPUT-15) — **bit-exact**.
4. **Depth-1 RMSE + Logloss** device-vs-CPU tree oracle — `<=1e-5` (Cosine score,
   first-order `calc_average` leaves).
5. **Steps 2–4 are ALL BLOCKING** — a failure halts the notebook so no fast-but-wrong
   speed number is ever reported (T-10-25).
6. Only then: **warm one untimed fit → time train-only**, draining CubeCL's lazy queue
   with a read-back/predict before stopping the clock → device vs host-CPU baseline
   (+ vs official CatBoost `task_type='GPU'` where a comparable depth-1 config exists).
7. Structured report → recorded by the human into `bench/RESULTS.md`.

## D-10-09 ESCALATION (read before trusting any depth-1 speed number)

> **Depth-1 device >= CPU is achievable ONLY at large n (~1e5–1e6+ rows); at small n it
> is physically infeasible regardless of optimization** — a depth-1 stump is the single
> most launch-overhead-bound workload in the milestone (a handful of kernel launches over
> trivial per-object work). The CPU grows it in microseconds; no fusion/residency makes a
> GPU launch + driver round-trip competitive at small n. **This is physics, not a tuning
> gap.** BENCH-02's depth-1 speed bar is therefore PINNED to the large-n synthetic
> workload (D-06 `SPEED_CONFIG`, ~1e6×50, tunable above break-even). **The Kaggle run is
> the arbiter of the crossover.** If device still loses even at large n after fused/
> resident optimization, record the measured crossover (or its absence) and let the user
> decide whether D-10-09 stands for depth-1 (depth-6 in Phase 11 is where device
> dominance is unambiguous). **Do NOT fabricate numbers** — every measured cell below is
> filled by the actual run.

## Step 0 — build the `--features cuda` wheel + confirm CUDA

Repo is expected checked out at `REPO`. Adjust the path for the Kaggle working dir.

In [ ]:
import os, subprocess, sys, time, json
import numpy as np

# --- repo layout (adjust REPO for the Kaggle working dir) -----------------------------
REPO = os.environ.get('CATBOOST_RS_REPO', os.getcwd())
BENCH = os.path.join(REPO, 'bench')
FIXTURES = os.path.join(BENCH, 'fixtures')
sys.path.insert(0, BENCH)
import generator  # D-06 single source: correctness fixture AND large-n speed workload

def run(cmd, cwd=REPO, check=True):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    sys.stdout.write(r.stdout);  sys.stderr.write(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {" ".join(cmd)}')
    return r

# Build the CUDA wheel (device path) and install it.
run(['maturin', 'build', '--release', '--no-default-features', '--features', 'cuda',
     '-m', 'crates/cb-python/Cargo.toml'])
# pip-install the freshly built wheel (newest by mtime under target/wheels).
wheels = sorted(
    (os.path.join(dp, f) for dp, _, fs in os.walk(os.path.join(REPO, 'target', 'wheels'))
     for f in fs if f.endswith('.whl')),
    key=os.path.getmtime)
assert wheels, 'no wheel built under target/wheels'
run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', wheels[-1]])

In [ ]:
# nvidia-smi — CUDA MUST be active or the whole run is meaningless.
smi = run(['nvidia-smi'], check=False)
assert smi.returncode == 0, 'nvidia-smi failed: this notebook must run on a CUDA instance'
print('CUDA confirmed active.')

## Steps 2–4 — BLOCKING correctness gate

The standalone primitive + cindex oracles run as the in-tree Rust device tests under
`--features cuda` (the same oracles that run on ROCm in-env, now on CUDA). The depth-1
RMSE/Logloss oracle compares the device `fit()` against the committed CPU-path expected
values in `bench/fixtures/`. **Any failure raises and halts — the speed cells assert the
gate passed before timing anything (T-10-25).**

In [ ]:
# GATE flag: set True only if ALL correctness oracles pass. Speed cells assert this.
GATE_PASSED = False
gate_report = {}

# --- Step 2 + 3: standalone primitive + cindex oracles (Rust device tests) ------------
# These encode the <=1e-4 / bit-exact bars in-tree (SPIKE-REDUCTION.md reduce harness et al).
prim = run(['cargo', 'test', '--no-default-features', '--features', 'cuda',
            '-p', 'cb-backend', '--', '--nocapture',
            'scan', 'segmented', 'sort', 'reorder', 'reduce', 'cindex', 'update_part_props'],
           check=False)
# --nocapture surfaces the reduce-strategy variance/path + err lines
# (reduce_finalize_strategies_are_deterministic_and_report_path). Transcribe the CUDA
# err/ms for candidates (a) FixedOrderTree / (b) HostSum / (c) FixedPointAtomic into
# SPIKE-REDUCTION.md section 4 from this output.
gate_report['primitive_cindex_oracles'] = 'PASS' if prim.returncode == 0 else 'FAIL'
assert prim.returncode == 0, 'BLOCKING: primitive/cindex device oracles FAILED — halting.'

In [ ]:
# --- Step 4: depth-1 RMSE + Logloss device-vs-CPU-reference oracle (<=1e-5) ------------
# Load committed correctness fixtures + numpy CPU-path expected values (D-06 single source).
X = np.load(os.path.join(FIXTURES, 'X_small.npy'))
y_reg = np.load(os.path.join(FIXTURES, 'y_small_reg.npy'))
y_bin = np.load(os.path.join(FIXTURES, 'y_small_bin.npy'))
with open(os.path.join(FIXTURES, 'expected_depth1_tree.json')) as fh:
    expected = json.load(fh)
lr = expected['learning_rate']

from catboost_rs import CatBoostRegressor, CatBoostClassifier  # device wheel (cuda)

EPS = 1e-5

# RMSE depth-1: one tree, first-order calc_average leaves. Device predictions on the
# training set for a single depth-1 tree == the per-object leaf value; compare against
# the committed serial reference (leaf_left/leaf_right on the reference split).
reg = CatBoostRegressor(iterations=1, depth=1, learning_rate=lr, l2_leaf_reg=expected['l2'])
reg.fit(X, y_reg)
pred_reg = np.asarray(reg.predict(X), dtype=np.float64)
ref = expected['rmse']
col = X[:, ref['best_feature']].astype(np.float64)
ref_pred = np.where(col > ref['best_border'], ref['leaf_right'], ref['leaf_left'])
rmse_err = float(np.max(np.abs(pred_reg - ref_pred)))
print(f'depth-1 RMSE device-vs-reference max abs err = {rmse_err:.3e} (bar {EPS:.0e})')
gate_report['depth1_rmse_max_abs_err'] = rmse_err
assert rmse_err <= EPS, f'BLOCKING: depth-1 RMSE oracle {rmse_err:.3e} > {EPS:.0e} — halting.'

# Logloss depth-1: FIRST-ORDER calc_average leaves (NOT Newton der2 — Phase 11 / GPUT-07).
clf = CatBoostClassifier(iterations=1, depth=1, learning_rate=lr, l2_leaf_reg=expected['l2'],
                         loss_function='Logloss')
clf.fit(X, y_bin)
raw = np.asarray(clf.predict(X, prediction_type='RawFormulaVal'), dtype=np.float64)
refl = expected['logloss']
coll = X[:, refl['best_feature']].astype(np.float64)
ref_raw = np.where(coll > refl['best_border'], refl['leaf_right'], refl['leaf_left'])
ll_err = float(np.max(np.abs(raw - ref_raw)))
print(f'depth-1 Logloss device-vs-reference max abs err = {ll_err:.3e} (bar {EPS:.0e})')
gate_report['depth1_logloss_max_abs_err'] = ll_err
assert ll_err <= EPS, f'BLOCKING: depth-1 Logloss oracle {ll_err:.3e} > {EPS:.0e} — halting.'

GATE_PASSED = True
print('\n=== CORRECTNESS GATE PASSED — speed cells unlocked ===')

## Steps 5–6 — warm-run / JIT-excluded train-only speed (large-n workload ONLY)

Pinned to the D-06 large-n `SPEED_CONFIG` (~1e6×50) per the D-10-09 escalation above.
**Warm one untimed fit** (absorbs JIT), then **time train-only** and force a read-back/
predict to **drain CubeCL's lazy queue** before stopping the clock (else the timer
under-reports; CubeCL executes lazily — manual `05_lazy_execution.md`,
`11_launch_overhead_and_transfers.md`).

In [ ]:
assert GATE_PASSED, 'REFUSING to time speed: correctness gate did not pass (T-10-25).'

# Large-n workload from the SAME seeded generator that produced the correctness fixture.
Xl, yl = generator.generate(**generator.SPEED_CONFIG)
print(f'speed workload: {Xl.shape[0]} rows x {Xl.shape[1]} features (D-06 SPEED_CONFIG)')

ITERS = 100  # depth-1 boosting rounds; tune so the device histogram amortizes launch cost

def timed_fit(make_model):
    m = make_model()
    m.fit(Xl, yl)               # WARM (untimed) — absorbs JIT / first-launch compile
    m2 = make_model()
    t0 = time.time()
    m2.fit(Xl, yl)
    _ = np.asarray(m2.predict(Xl[:1024]))  # DRAIN the lazy queue before stopping the clock
    return time.time() - t0

device_s = timed_fit(lambda: CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1))
print(f'device (cuda) train-only wall-clock: {device_s:.4f}s')
gate_report['speed_device_s'] = device_s

In [ ]:
# Host-CPU baseline + official CatBoost GPU, where available on the box.
# NOTE: the catboost_rs CPU baseline requires a SEPARATE cpu-feature wheel (features are
# compile-time, CLAUDE.md). Build+install it in a fresh cell / kernel if comparing on the
# same box; otherwise record the CPU number from a cpu-wheel run and paste into RESULTS.md.
cpu_s = None      # <-- fill from a cpu-feature wheel run (see note above)
catboost_gpu_s = None
try:
    import catboost as _cb
    m = _cb.CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1,
                              task_type='GPU', verbose=False)
    m.fit(Xl, yl); m2 = _cb.CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1,
                                              task_type='GPU', verbose=False)
    t0 = time.time(); m2.fit(Xl, yl); _ = m2.predict(Xl[:1024])
    catboost_gpu_s = time.time() - t0
    print(f'official CatBoost GPU train-only: {catboost_gpu_s:.4f}s')
except Exception as e:
    print('official CatBoost GPU comparison skipped:', e)
gate_report['speed_cpu_s'] = cpu_s
gate_report['speed_catboost_gpu_s'] = catboost_gpu_s

### Fill `SPIKE-REDUCTION.md` §4 (CUDA-only reduce rows) from this run

The `--nocapture` reduce output above reports, per finalize strategy, the run-to-run
spread + path (and `abs_div` vs the f64 baseline). Transcribe the CUDA `err (<=1e-4)` and
`ms` for the three reduce candidates into `SPIKE-REDUCTION.md` §4:

| Candidate | CUDA err (<=1e-4) | CUDA ms |
|-----------|-------------------|---------|
| (a) Fixed-order tree reduce | _paste_ | _paste_ |
| (b) Block-then-host-sum     | _paste_ | _paste_ |
| (c) Fixed-point u64 atomics | _paste_ | _paste_ |

These are **not fabricated** — they come straight from the notebook's device run.

## Step 7 — structured report

Copy the printed rows into `bench/RESULTS.md` (correctness pass/fail + speed) and fill the
`SPIKE-REDUCTION.md` §4 CUDA `err`/`ms` rows from the reduce-strategy oracle output above.
Correctness rows come FIRST; the speed row is only meaningful because the gate passed.

In [ ]:
print('================ CUDA ORACLE STRUCTURED REPORT ================')
print('CORRECTNESS (blocking gate):')
print(f"  primitive + cindex oracles : {gate_report.get('primitive_cindex_oracles')}")
print(f"  depth-1 RMSE   max|err|     : {gate_report.get('depth1_rmse_max_abs_err'):.3e}  (bar 1e-5)")
print(f"  depth-1 Logloss max|err|    : {gate_report.get('depth1_logloss_max_abs_err'):.3e}  (bar 1e-5)")
print(f"  GATE PASSED                 : {GATE_PASSED}")
print('SPEED (large-n SPEED_CONFIG, warm-run/JIT-excluded, queue drained):')
print(f"  device (cuda)   train-only  : {gate_report.get('speed_device_s')} s")
print(f"  host-CPU baseline           : {gate_report.get('speed_cpu_s')} s  (cpu-wheel; see note)")
print(f"  official CatBoost GPU        : {gate_report.get('speed_catboost_gpu_s')} s")
print('D-10-09: depth-1 device>=CPU only at large n; small n physically infeasible.')
print('==============================================================')
print(json.dumps(gate_report, indent=2, default=str))

## Phase 11 — Depth-6 CUDA-authoritative gate (GPUT-14 / GPUT-06)

**Human-gated, Kaggle-CUDA-authoritative.** These depth-6 cells close Phase 11 on the sole
authoritative GPU oracle. ROCm in-env (Plans 02–04) is smoke-only — **not** the gate.

**Bars.** The **device path** is gated at **ε=1e-4** (the standing GPUT-14 bar); the **CPU
reference** stays the **≤1e-5** oracle of record (Plan 01 `expected_depth6_tree.json`,
byte-unchanged — D-04). **Correctness is BLOCKING before any speed number** (no fabricated
numbers).

This section is **self-contained**: it re-imports the device wheel and re-loads the
committed workload, so it can be run independently of the depth-1 cells above.

What runs:
1. **Gate A (CUDA-alone, byte-unchanged fixture, BLOCKING)** — the device single depth-6
   tree (`iterations=1`, the fixture's pinned A1 config) vs the committed Plan-01 CPU
   reference, compared **base-free** (prediction differences, so the RMSE mean prior /
   Logloss logit prior cancels): `max|Δdevice − Δcpu| ≤ 1e-4` for **RMSE and Logloss**.
2. **Gate B (device full-run vs the Rust CPU-feature wheel)** — the literal "device vs Rust
   CPU path" over a multi-iteration depth-6 run. CPU-path predictions come from a **separate
   cpu-feature wheel** (features are compile-time, CLAUDE.md), saved to
   `cpu_pred_{rmse,logloss}.npy`; run the cpu helper cell (in a fresh cpu-wheel kernel)
   first. Absent ⇒ reported **PENDING**, never fabricated.
3. **Per-tree split-agreement + run-to-run spread diagnostic (GPUT-06 / D-05)** — the
   in-tree `cb-backend` CUDA grow-loop oracles hard-assert the per-level split
   `(feature,bin)` sequence + per-object `leaf_of` **exactly** vs the CPU greedy reference,
   plus a device run-to-run spread curve over growing prefixes that localizes any
   compounding split-flip to the tree where the spread first steps up (Pitfall 4).

> **Logloss margin.** `CatBoostClassifier.predict` returns class labels (no
> `prediction_type` kwarg); the raw margin is `logit(predict_proba[:,1])`. Base-free
> centering makes the prior cancel, so this is the correct device-vs-CPU margin comparison.

In [ ]:
# ---- Phase 11 depth-6: load the committed byte-unchanged workload + fixture (D-04) -----
import os, json, time
import numpy as np
# (REPO/BENCH/FIXTURES/generator/run are defined in the Step-0 cells; re-derive if this
# section is run standalone.)
try:
    FIXTURES
except NameError:
    REPO = os.environ.get('CATBOOST_RS_REPO', os.getcwd())
    BENCH = os.path.join(REPO, 'bench'); FIXTURES = os.path.join(BENCH, 'fixtures')
    import sys; sys.path.insert(0, BENCH)
    import generator

X6 = np.load(os.path.join(FIXTURES, 'X_small.npy'))          # correctness design matrix
y6_reg = np.load(os.path.join(FIXTURES, 'y_small_reg.npy'))
y6_bin = np.load(os.path.join(FIXTURES, 'y_small_bin.npy'))
with open(os.path.join(FIXTURES, 'expected_depth6_tree.json')) as fh:
    d6 = json.load(fh)
d6cfg = d6['config']
LR6, L2_6, DEPTH6 = d6cfg['learning_rate'], d6cfg['l2_leaf_reg'], d6cfg['depth']
assert d6cfg['leaf_estimation_iterations'] == 1, 'fixture must pin A1 (iterations==1)'
assert DEPTH6 == 6, 'this is the depth-6 gate'

EPS_DEVICE = 1e-4    # GPUT-14 device bar
EPS_CPUREF = 1e-5    # CPU reference-of-record bar (Plan 01)
ITERS_D6   = 200     # full-run length for Gate B + the per-tree diagnostic (hundreds of trees)

def _centered(a):
    a = np.asarray(a, dtype=np.float64)
    return a - a.mean()

# CPU reference-of-record contribution for a SINGLE depth-6 tree (iterations=1):
#   pred_i = base + LR6 * leaf_delta[leaf_of_i]   (leaf_values are RAW deltas; the boosting
# loop multiplies by LR6). Base-free centering cancels the unknown RMSE mean prior /
# Logloss logit prior, so this compares the device tree to the CPU reference WITHOUT
# retraining the reference (D-04 byte-unchanged).
def cpu_ref_centered(arm):
    leaf_of = np.asarray(d6[arm]['leaf_of'], dtype=np.int64)
    leaf_values = np.asarray(d6[arm]['leaf_values'], dtype=np.float64)
    return _centered(LR6 * leaf_values[leaf_of])

def logit_margin(clf, X):
    p = np.asarray(clf.predict_proba(X), dtype=np.float64)[:, 1]
    p = np.clip(p, 1e-12, 1.0 - 1e-12)
    return np.log(p) - np.log1p(-p)

print(f'depth-6 workload: {X6.shape[0]} x {X6.shape[1]}  lr={LR6} l2={L2_6} depth={DEPTH6}')
print('CPU reference-of-record: committed expected_depth6_tree.json (Plan 01, D-04 byte-unchanged).')

In [ ]:
# ---- Gate B CPU-path predictions: RUN THIS IN A SEPARATE cpu-feature-wheel kernel -------
# Backend selection is COMPILE-TIME (CLAUDE.md): the CPU path needs its OWN wheel and cannot
# be imported in the same process as the cuda wheel. In a FRESH Kaggle kernel:
#   maturin build --release --no-default-features --features cpu -m crates/cb-python/Cargo.toml
#   pip install --force-reinstall <that wheel>
# then set BUILD_CPU_REF=True and run THIS cell to emit cpu_pred_{rmse,logloss}.npy, which
# the Gate B cell below loads. Do NOT run this in the CUDA kernel (wrong wheel).
BUILD_CPU_REF = False
if BUILD_CPU_REF:
    from catboost_rs import CatBoostRegressor as _R, CatBoostClassifier as _C
    r = _R(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6)
    r.fit(X6, y6_reg)
    np.save(os.path.join(FIXTURES, 'cpu_pred_rmse.npy'),
            np.asarray(r.predict(X6), dtype=np.float64))
    c = _C(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6,
           loss_function='Logloss')
    c.fit(X6, y6_bin)
    np.save(os.path.join(FIXTURES, 'cpu_pred_logloss.npy'), logit_margin(c, X6))
    print('wrote cpu_pred_{rmse,logloss}.npy (Rust CPU path) for Gate B')
else:
    print('BUILD_CPU_REF=False — Gate B loads cpu_pred_*.npy if a cpu-wheel run produced them.')

In [ ]:
# ============ BLOCKING depth-6 final-prediction ε=1e-4 gate (GPUT-14) ==================
from catboost_rs import CatBoostRegressor, CatBoostClassifier   # cuda wheel (device path)
GATE_D6_PASSED = False
d6_report = {}

# ---- Gate A: device single depth-6 tree vs the committed CPU reference (base-free) ------
# iterations=1 == the fixture's pinned A1 config; the CPU reference is NOT retrained (D-04).
regA = CatBoostRegressor(iterations=1, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6)
regA.fit(X6, y6_reg)
errA_rmse = float(np.max(np.abs(_centered(regA.predict(X6)) - cpu_ref_centered('rmse'))))
print(f'[Gate A] depth-6 RMSE    device-vs-CPU-ref max|delta| = {errA_rmse:.3e} '
      f'(device bar {EPS_DEVICE:.0e}; CPU ref {EPS_CPUREF:.0e})')

clfA = CatBoostClassifier(iterations=1, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6,
                          loss_function='Logloss')
clfA.fit(X6, y6_bin)
errA_ll = float(np.max(np.abs(_centered(logit_margin(clfA, X6)) - cpu_ref_centered('logloss'))))
print(f'[Gate A] depth-6 Logloss device-vs-CPU-ref max|delta| = {errA_ll:.3e} '
      f'(device bar {EPS_DEVICE:.0e}; CPU ref {EPS_CPUREF:.0e})')
d6_report['gateA_rmse_max_abs_err'] = errA_rmse
d6_report['gateA_logloss_max_abs_err'] = errA_ll
assert errA_rmse <= EPS_DEVICE, f'BLOCKING: depth-6 RMSE Gate A {errA_rmse:.3e} > {EPS_DEVICE:.0e}'
assert errA_ll   <= EPS_DEVICE, f'BLOCKING: depth-6 Logloss Gate A {errA_ll:.3e} > {EPS_DEVICE:.0e}'

# ---- Gate B: device full-run vs the Rust CPU-feature-wheel predictions (literal CPU path)
def _gateB(arm, fit_and_predict):
    cpu_path = os.path.join(FIXTURES, f'cpu_pred_{arm}.npy')
    if not os.path.exists(cpu_path):
        print(f'[Gate B] {arm}: PENDING — cpu_pred_{arm}.npy absent (run the cpu-wheel helper '
              f'cell in a separate kernel). NOT fabricated.')
        d6_report[f'gateB_{arm}'] = 'PENDING'
        return
    dev = fit_and_predict()
    cpu = np.load(cpu_path)
    err = float(np.max(np.abs(np.asarray(dev, dtype=np.float64) - cpu)))
    print(f'[Gate B] depth-6 {arm} device-vs-CPU-wheel (full {ITERS_D6}-iter run) '
          f'max|delta| = {err:.3e} (bar {EPS_DEVICE:.0e})')
    d6_report[f'gateB_{arm}_max_abs_err'] = err
    assert err <= EPS_DEVICE, f'BLOCKING: depth-6 {arm} Gate B {err:.3e} > {EPS_DEVICE:.0e}'

def _fit_reg():
    m = CatBoostRegressor(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6)
    m.fit(X6, y6_reg); return m.predict(X6)
def _fit_clf():
    m = CatBoostClassifier(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6,
                           l2_leaf_reg=L2_6, loss_function='Logloss')
    m.fit(X6, y6_bin); return logit_margin(m, X6)
_gateB('rmse', _fit_reg)
_gateB('logloss', _fit_clf)

GATE_D6_PASSED = True    # Gate A (blocking) passed; Gate B pass/PENDING recorded above.
print('\n=== depth-6 Gate A PASSED (device <=1e-4 vs the CPU reference-of-record) ===')
print('    Gate B (full-run vs the cpu wheel) recorded above (PENDING if cpu preds absent).')

In [ ]:
# ========== Per-tree split-agreement + run-to-run spread diagnostic (GPUT-06 / D-05) ====
assert GATE_D6_PASSED, 'run the depth-6 gate cell first'

# (1) SPLIT-AGREEMENT — the in-tree cb-backend CUDA grow-loop oracles hard-assert the
# per-level split (feature,bin) sequence + per-object leaf_of EXACTLY vs the CPU greedy
# reference (depth6_rmse_grow_matches_cpu / newton_leaf_matches_cpu) and prove zero
# run-to-run spread over 32 launches of the maximal-contention 64-partition fixed-point
# accumulator (partition_hist_reduce_zero_spread, GPUT-06). Running them under
# --features cuda is the per-tree split-agreement evidence on the authoritative oracle;
# a failure prints the first divergent LEVEL in the --nocapture output.
split_oracle = run(['cargo', 'test', '--no-default-features', '--features', 'cuda',
                    '-p', 'cb-backend', '--', '--nocapture',
                    'depth6_rmse_grow_matches_cpu', 'newton_leaf_matches_cpu',
                    'partition_hist_reduce_zero_spread'], check=False)
d6_report['split_agreement_oracle'] = 'PASS' if split_oracle.returncode == 0 else 'FAIL'
assert split_oracle.returncode == 0, ('BLOCKING: CUDA grow-loop split-agreement / zero-spread '
                                      'oracle FAILED — see the first divergent level above.')

# (2) RUN-TO-RUN SPREAD vs prefix length — localizes a compounding split-flip to the tree
# where the spread first STEPS UP (Pitfall 4). The LOCKED deterministic reduce (Plans 02/03)
# ⇒ spread should stay 0.0 across the whole run; a step-change pinpoints the originating
# tree. Device wheel only (no cpu wheel needed).
PREFIXES = [1, 2, 4, 8, 16, 32, 64, 128, ITERS_D6]
REPEATS = 3
spread_curve = {}
for t in PREFIXES:
    preds = []
    for _ in range(REPEATS):
        m = CatBoostRegressor(iterations=t, depth=DEPTH6, learning_rate=LR6, l2_leaf_reg=L2_6)
        m.fit(X6, y6_reg)
        preds.append(np.asarray(m.predict(X6), dtype=np.float64))
    stack = np.stack(preds, axis=0)
    spread = float(np.max(np.max(stack, axis=0) - np.min(stack, axis=0)))
    spread_curve[t] = spread
    print(f'  prefix={t:>4} trees  run-to-run spread = {spread:.3e}')
d6_report['run_to_run_spread_curve'] = spread_curve
first_div = next((t for t in PREFIXES if spread_curve[t] > EPS_DEVICE), None)
d6_report['first_divergent_prefix'] = first_div
if first_div is None:
    print(f'\nNo compounding drift: run-to-run spread <= {EPS_DEVICE:.0e} across all '
          f'{ITERS_D6} trees (SC-3 / GPUT-06 — no split flips compounding).')
else:
    print(f'\nDrift localized: spread first exceeds {EPS_DEVICE:.0e} at prefix {first_div} '
          f'trees — inspect that tree (Pitfall 4 step-change).')

## Phase 11 — Depth-6 speed (BENCH-02): device vs host-CPU vs official CatBoost GPU

**Warm-run / JIT-excluded / train-only, lazy queue drained.** Only meaningful because the
depth-6 correctness gate above PASSED (`GATE_D6_PASSED`) — correctness is BLOCKING before
any speed number (T-10-25). Large-n `SPEED_CONFIG` (~1e6×50) is regenerated on the fly from
the same seeded generator (D-06); the committed `X_depth6_speed.npy` (10k×50) is the small
representative. **Depth-6 is where device dominance is unambiguous** (D-10-09 — unlike the
launch-overhead-bound depth-1 stump).

Both arms are timed: depth-6 **RMSE** and depth-6 **Logloss**. The host-CPU
`catboost_rs` baseline needs a **separate cpu-feature wheel** (compile-time features) — its
number is filled from a cpu-wheel run (see the Gate-B helper note); do NOT run the cpu
`catboost_rs` baseline in the CUDA kernel. Official CatBoost GPU (`task_type='GPU'`, depth=6)
runs inline where available. **Do NOT fabricate numbers** — every measured cell is the run's.

In [ ]:
# ============ depth-6 device train-only wall-clock (warm-run/JIT-excluded) =============
assert GATE_D6_PASSED, 'REFUSING to time depth-6 speed: correctness gate not passed (T-10-25).'

# Large-n depth-6 workload from the SAME seeded generator (D-06); regenerated on the fly.
X6l, y6l = generator.generate(**generator.SPEED_CONFIG)
y6l_bin = generator.binary_target(X6l, generator.SPEED_CONFIG['seed'])
print(f'depth-6 speed workload: {X6l.shape[0]} x {X6l.shape[1]} (SPEED_CONFIG, regenerated)')

def timed_fit6(make_model, y):
    m = make_model(); m.fit(X6l, y)                 # WARM (untimed) — absorbs JIT / first launch
    m2 = make_model()
    t0 = time.time(); m2.fit(X6l, y)
    _ = np.asarray(m2.predict(X6l[:1024]))          # DRAIN the lazy CubeCL queue before stop
    return time.time() - t0

dev_rmse_s = timed_fit6(
    lambda: CatBoostRegressor(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6), y6l)
dev_ll_s = timed_fit6(
    lambda: CatBoostClassifier(iterations=ITERS_D6, depth=DEPTH6, learning_rate=LR6,
                               loss_function='Logloss'), y6l_bin)
print(f'device (cuda) depth-6 RMSE    train-only: {dev_rmse_s:.4f}s')
print(f'device (cuda) depth-6 Logloss train-only: {dev_ll_s:.4f}s')
d6_report['speed_device_rmse_s'] = dev_rmse_s
d6_report['speed_device_logloss_s'] = dev_ll_s

In [ ]:
# ---- host-CPU baseline (cpu-feature wheel, separate kernel) + official CatBoost GPU -----
# The catboost_rs CPU baseline requires its OWN cpu-feature wheel (compile-time features);
# fill these from a cpu-wheel depth-6 run (do NOT run the cpu baseline in the CUDA kernel).
cpu_rmse_s = None        # <-- fill from a cpu-feature wheel depth-6 RMSE run
cpu_logloss_s = None     # <-- fill from a cpu-feature wheel depth-6 Logloss run
cb_gpu_rmse_s = cb_gpu_logloss_s = None
try:
    import catboost as _cb
    def _t(make, y):
        m = make(); m.fit(X6l, y); m2 = make()
        t0 = time.time(); m2.fit(X6l, y); _ = m2.predict(X6l[:1024]); return time.time() - t0
    cb_gpu_rmse_s = _t(lambda: _cb.CatBoostRegressor(iterations=ITERS_D6, depth=DEPTH6,
                       learning_rate=LR6, task_type='GPU', verbose=False), y6l)
    cb_gpu_logloss_s = _t(lambda: _cb.CatBoostClassifier(iterations=ITERS_D6, depth=DEPTH6,
                          learning_rate=LR6, loss_function='Logloss',
                          task_type='GPU', verbose=False), y6l_bin)
    print(f'official CatBoost GPU depth-6 RMSE    : {cb_gpu_rmse_s:.4f}s')
    print(f'official CatBoost GPU depth-6 Logloss : {cb_gpu_logloss_s:.4f}s')
except Exception as e:
    print('official CatBoost GPU depth-6 comparison skipped:', e)
d6_report['speed_cpu_rmse_s'] = cpu_rmse_s
d6_report['speed_cpu_logloss_s'] = cpu_logloss_s
d6_report['speed_catboost_gpu_rmse_s'] = cb_gpu_rmse_s
d6_report['speed_catboost_gpu_logloss_s'] = cb_gpu_logloss_s

In [ ]:
# ================= PHASE 11 DEPTH-6 CUDA STRUCTURED REPORT (copy into RESULTS.md) =======
def _f(x):
    return f'{x:.3e}' if isinstance(x, float) else str(x)
print('============== PHASE 11 DEPTH-6 CUDA STRUCTURED REPORT ==============')
print('CORRECTNESS (blocking gate — device bar 1e-4; CPU reference-of-record 1e-5):')
print(f"  Gate A RMSE    device-vs-CPU-ref max|d| : {_f(d6_report.get('gateA_rmse_max_abs_err'))}")
print(f"  Gate A Logloss device-vs-CPU-ref max|d| : {_f(d6_report.get('gateA_logloss_max_abs_err'))}")
print(f"  Gate B RMSE    (full run vs cpu wheel)  : {d6_report.get('gateB_rmse_max_abs_err', d6_report.get('gateB_rmse', 'PENDING'))}")
print(f"  Gate B Logloss (full run vs cpu wheel)  : {d6_report.get('gateB_logloss_max_abs_err', d6_report.get('gateB_logloss', 'PENDING'))}")
print(f"  split-agreement oracle (CUDA)           : {d6_report.get('split_agreement_oracle')}")
print(f"  first divergent prefix (spread)         : {d6_report.get('first_divergent_prefix')}")
print(f"  GATE_D6_PASSED                          : {GATE_D6_PASSED}")
print('SPEED (depth-6, warm-run/JIT-excluded/train-only, SPEED_CONFIG):')
print(f"  device (cuda) RMSE / Logloss : {d6_report.get('speed_device_rmse_s')} / {d6_report.get('speed_device_logloss_s')} s")
print(f"  host-CPU      RMSE / Logloss : {d6_report.get('speed_cpu_rmse_s')} / {d6_report.get('speed_cpu_logloss_s')} s  (cpu wheel)")
print(f"  CatBoost GPU  RMSE / Logloss : {d6_report.get('speed_catboost_gpu_rmse_s')} / {d6_report.get('speed_catboost_gpu_logloss_s')} s")
print('Correctness rows FIRST; speed rows are only meaningful because the gate passed.')
print('====================================================================')
import json as _json
print(_json.dumps(d6_report, indent=2, default=str))